# Run stability and costs

Analyze the five saved Gemini runs without making new API calls.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [72]:
import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")

from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index, metadata_directory, pdf_directory,
    reference_datasets, review_records, case_study_filters, align_full_text_results,
)

load_environment()

import numpy as np
import os
import pandas as pd

from statsmodels.stats.inter_rater import fleiss_kappa

folder_path = data_directory()
metadata_path = metadata_directory("review")
result_path = results_directory("gemini")
screening_path = result_path + 'screening/'
eligibility_path = result_path + 'eligibility/'

In [65]:
csv_file_path = screening_path + "cost_summary.csv"
df = pd.read_csv(csv_file_path)
df

,run_id,stage,model,records_processed,api_calls,input_tokens,output_tokens,execution_time_sec,theoretical_time_sec,cost_input_usd,cost_output_usd,cost_usd
0,run_1,screening,gemini-2.5-flash,145,10,42634,9749,293.938,240.0,0.012790,0.024372,0.037163
1,run_1,harmonization,gemini-2.5-pro,145,1,2591,1003,55.328,NaN,0.003239,0.010030,0.013269
2,run_2,screening,gemini-2.5-flash,145,10,42634,9786,313.828,240.0,0.012790,0.024465,0.037255
3,run_2,harmonization,gemini-2.5-pro,145,1,2607,1018,78.422,NaN,0.003259,0.010180,0.013439
4,run_3,screening,gemini-2.5-flash,145,10,42634,9623,306.422,240.0,0.012790,0.024057,0.036848
5,run_3,harmonization,gemini-2.5-pro,145,1,2606,1013,56.453,NaN,0.003257,0.010130,0.013388
6,run_4,screening,gemini-2.5-flash,145,10,42634,9534,312.281,240.0,0.012790,0.023835,0.036625
7,run_4,harmonization,gemini-2.5-pro,145,1,2607,1014,45.422,NaN,0.003259,0.010140,0.013399
8,run_5,screening,gemini-2.5-flash,145,10,42634,10406,301.641,240.0,0.012790,0.026015,0.038805
9,run_5,harmonization,gemini-2.5-pro,145,1,2595,1030,46.468,NaN,0.003244,0.010300,0.013544


In [66]:
metrics = [
    "records_processed",
    "api_calls",
    "input_tokens",
    "output_tokens",
    "execution_time_sec",
    "cost_input_usd",
    "cost_output_usd",
    "cost_usd"
]

summary = df.groupby(["stage", "model"])[metrics].agg(["mean", "std"])
summary["n_runs"] = df.groupby(["stage", "model"])["run_id"].nunique()

formatted_summary = pd.DataFrame(index=summary.index)
formatted_summary["n_runs"] = summary["n_runs"]

for col in metrics:
    mean_series = summary[col]["mean"]
    std_series = summary[col]["std"].fillna(0)

    if "tokens" in col or "records" in col or "calls" in col:
        formatted_summary[col] = mean_series.round(0).astype(int).astype(str) + " ± " + std_series.round(0).astype(int).astype(str)
    elif "cost" in col:
        formatted_summary[col] = "$" + mean_series.round(4).astype(str) + " ± " + std_series.round(4).astype(str)
    else:
        formatted_summary[col] = mean_series.round(1).astype(str) + " ± " + std_series.round(1).astype(str)

print("=== AGGREGATED METRICS ACROSS MULTIPLE RUNS ===")
print(formatted_summary.reset_index().to_markdown(index=False))

=== AGGREGATED METRICS ACROSS MULTIPLE RUNS ===
| stage         | model            |   n_runs | records_processed   | api_calls   | input_tokens   | output_tokens   | execution_time_sec   | cost_input_usd   | cost_output_usd   | cost_usd         |
|:--------------|:-----------------|---------:|:--------------------|:------------|:---------------|:----------------|:---------------------|:-----------------|:------------------|:-----------------|
| harmonization | gemini-2.5-pro   |        5 | 145 ± 0             | 1 ± 0       | 2601 ± 8       | 1016 ± 10       | 56.4 ± 13.3          | $0.0033 ± 0.0    | $0.0102 ± 0.0001  | $0.0134 ± 0.0001 |
| screening     | gemini-2.5-flash |        5 | 145 ± 0             | 10 ± 0      | 42634 ± 0      | 9820 ± 343      | 305.6 ± 8.1          | $0.0128 ± 0.0    | $0.0245 ± 0.0009  | $0.0373 ± 0.0009 |


In [67]:
dfs = []
for i in range(1, 6):
    file_path = os.path.join(screening_path, f'run_{i}_records.csv')
    df = pd.read_csv(file_path)
    dfs.append(df)

df_compare = pd.DataFrame({
    'run_1': dfs[0]['Include'],
    'run_2': dfs[1]['Include'],
    'run_3': dfs[2]['Include'],
    'run_4': dfs[3]['Include'],
    'run_5': dfs[4]['Include']
})

df_compare['agreement'] = df_compare.eq(df_compare.iloc[:, 0], axis=0).all(axis=1)
discrepancies = df_compare[~df_compare['agreement']]

print(f"Record with discrepancies: {len(discrepancies)}")
print(discrepancies)

table = []
for idx, row in df_compare[['run_1', 'run_2', 'run_3', 'run_4', 'run_5']].iterrows():
    yes_count = (row == 'yes').sum()
    no_count = (row == 'no').sum()
    table.append([yes_count, no_count])

kappa = fleiss_kappa(table, method='fleiss')

print(f"Fleiss' Kappa: {kappa:.4f}")

Record with discrepancies: 3
    run_1 run_2 run_3 run_4 run_5  agreement
65     no   yes   yes   yes    no      False
126    no   yes   yes   yes    no      False
130    no   yes   yes   yes    no      False
Fleiss' Kappa: 0.9726


In [69]:
def replace_long_datasets(text):
    if not text:
        return text
    return ';'.join(['self-created' if len(p) > 20 else p for p in text.split(';')])

dfs = []
for i in range(1, 6):
    file_path = os.path.join(screening_path, f'run_{i}_records.csv')
    df = pd.read_csv(file_path)
    dfs.append(df)

df_ds = pd.DataFrame({
    f'run_{i}': dfs[i-1]['Data Source'] for i in range(1, 6)
})

for col in df_ds.columns:
    df_ds[col] = (df_ds[col]
                  .fillna('')
                  .astype(str)
                  .str.lower()
                  .str.replace(r'\s*;\s*', ';', regex=True)
                  .str.strip())
    df_ds[col] = df_ds[col].apply(replace_long_datasets)

df_ds['agreement'] = df_ds.eq(df_ds.iloc[:, 0], axis=0).all(axis=1)
df_ds.insert(0, 'DOI', dfs[0]['DOI'])
discrepancies = df_ds[~df_ds['agreement']].drop(columns=['agreement'])

print(f"Records with discrepancies: {len(discrepancies)}")
discrepancies

Records with discrepancies: 4


,DOI,run_1,run_2,run_3,run_4,run_5
26,10.1109/CVPR.2019.00039,multitask-aqa,,,,
65,10.1109/CVPR52733.2024.01744,,mtl-aqa;finegym,mtl-aqa;finegym,mtl-aqa;finegym,
126,10.1016/j.patcog.2017.12.007,,msr-action3d;msrc-12,msr-action3d;msrc-12,msr-action3d;msrc-12,
130,10.1109/SSCI.2017.8285270,,self-created,self-created,self-created,


In [88]:
csv_file_path = eligibility_path + "cost_summary.csv"
df = pd.read_csv(csv_file_path)
df

,run_id,stage,model,records_processed,api_calls,input_tokens,output_tokens,embedding_tokens,index_time_sec,extraction_time_sec,execution_time_sec,cost_input_usd,cost_output_usd,cost_embedding_usd,cost_usd
0,run_1,eligibility,gemini-2.5-flash,21,20,46083,229,302586.0,47.016,110.906,157.922,0.013825,0.000572,0.060517,0.074915
1,run_1,harmonization,gemini-2.5-pro,21,1,514,195,NaN,NaN,NaN,17.719,0.000642,0.001950,NaN,0.002593
2,run_2,eligibility,gemini-2.5-flash,21,20,46083,229,302586.0,51.218,117.282,168.500,0.013825,0.000572,0.060517,0.074915
3,run_2,harmonization,gemini-2.5-pro,21,1,514,194,NaN,NaN,NaN,27.750,0.000642,0.001940,NaN,0.002582
4,run_3,eligibility,gemini-2.5-flash,21,20,46083,229,302586.0,49.891,107.172,157.063,0.013825,0.000572,0.060517,0.074915
5,run_3,harmonization,gemini-2.5-pro,21,1,514,194,NaN,NaN,NaN,33.015,0.000642,0.001940,NaN,0.002582
6,run_4,eligibility,gemini-2.5-flash,21,20,46083,218,302586.0,52.969,121.187,174.156,0.013825,0.000545,0.060517,0.074887
7,run_4,harmonization,gemini-2.5-pro,21,1,505,184,NaN,NaN,NaN,19.547,0.000631,0.001840,NaN,0.002471
8,run_5,eligibility,gemini-2.5-flash,21,20,46083,219,302586.0,48.656,114.657,163.313,0.013825,0.000548,0.060517,0.074890
9,run_5,harmonization,gemini-2.5-pro,21,1,506,187,NaN,NaN,NaN,21.172,0.000633,0.001870,NaN,0.002502


In [91]:
metrics = [
    "records_processed",
    "api_calls",
    "input_tokens",
    "output_tokens",
    "embedding_tokens",
    "index_time_sec",
    "extraction_time_sec",
    "execution_time_sec",
    "theoretical_time_sec",
    "cost_input_usd",
    "cost_output_usd",
    "cost_embedding_usd",
    "cost_usd"
]

metrics = [m for m in metrics if m in df.columns]

for col in metrics:
    df[col] = pd.to_numeric(df[col], errors='coerce')

if 'embedding_tokens' in df.columns and 'cost_embedding_usd' in df.columns:
    has_emb = df['embedding_tokens'] > 0
    if has_emb.any():
        df_emb = df[has_emb].copy()
        df_emb['stage'] = 'embedding'
        df_emb['model'] = 'gemini-embedding-2'

        gen_cols = ['api_calls', 'input_tokens', 'output_tokens', 'extraction_time_sec', 'cost_input_usd', 'cost_output_usd', 'theoretical_time_sec']
        for c in gen_cols:
            if c in df_emb.columns:
                df_emb[c] = np.nan

        emb_cols = ['embedding_tokens', 'index_time_sec', 'cost_embedding_usd']
        for c in emb_cols:
            if c in df.columns:
                df.loc[has_emb, c] = np.nan

        if 'cost_usd' in df.columns:
            df_emb['cost_usd'] = df_emb['cost_embedding_usd']
            df.loc[has_emb, 'cost_usd'] = df.loc[has_emb, 'cost_usd'] - df_emb['cost_embedding_usd'].values

        df = pd.concat([df, df_emb], ignore_index=True)

summary = df.groupby(["stage", "model"])[metrics].agg(["mean", "std"])
summary["n_runs"] = df.groupby(["stage", "model"])["run_id"].nunique()

formatted_summary = pd.DataFrame(index=summary.index)
formatted_summary["n_runs"] = summary["n_runs"]

for col in metrics:
    mean_val = summary[col]["mean"].fillna(0)
    std_val = summary[col]["std"].fillna(0)

    formatted_col = []
    for m, s in zip(mean_val, std_val):
        if np.isclose(m, 0) and np.isclose(s, 0):
            formatted_col.append("-")
        else:
            if "tokens" in col or "records" in col or "calls" in col:
                formatted_col.append(f"{int(round(m))} ± {int(round(s))}")
            elif "cost" in col:
                formatted_col.append(f"${m:.4f} ± {s:.4f}")
            else:
                formatted_col.append(f"{m:.2f} ± {s:.2f}")

    formatted_summary[col] = formatted_col

print("=== AGGREGATED METRICS ACROSS MULTIPLE RUNS ===")
print(formatted_summary.reset_index().to_markdown(index=False))

=== AGGREGATED METRICS ACROSS MULTIPLE RUNS ===
| stage         | model              |   n_runs | records_processed   | api_calls   | input_tokens   | output_tokens   | embedding_tokens   | index_time_sec   | extraction_time_sec   | execution_time_sec   | cost_input_usd   | cost_output_usd   | cost_embedding_usd   | cost_usd         |
|:--------------|:-------------------|---------:|:--------------------|:------------|:---------------|:----------------|:-------------------|:-----------------|:----------------------|:---------------------|:-----------------|:------------------|:---------------------|:-----------------|
| eligibility   | gemini-2.5-flash   |        5 | 21 ± 0              | 20 ± 0      | 46083 ± 0      | 225 ± 6         | -                  | -                | 114.24 ± 5.45         | 164.19 ± 7.22        | $0.0138 ± 0.0000 | $0.0006 ± 0.0000  | -                    | $0.0144 ± 0.0000 |
| embedding     | gemini-embedding-2 |        5 | 21 ± 0              | -           |

In [85]:
def replace_long_datasets(text):
    if not text:
        return text
    return ';'.join(['self-created' if len(p) > 20 else p for p in text.split(';')])

dfs = []
for i in range(1, 6):
    file_path = os.path.join(eligibility_path, f'run_{i}_records.csv')
    df = pd.read_csv(file_path)
    dfs.append(df)

df_ds = pd.DataFrame({
    f'run_{i}': dfs[i-1]['Data Source Harmonized'] for i in range(1, 6)
})

for col in df_ds.columns:
    df_ds[col] = (df_ds[col]
                  .fillna('')
                  .astype(str)
                  .str.lower()
                  .str.replace(r'\s*;\s*', ';', regex=True)
                  .str.strip())
    df_ds[col] = df_ds[col].apply(replace_long_datasets)

df_ds['agreement'] = df_ds.eq(df_ds.iloc[:, 0], axis=0).all(axis=1)
#df_ds.insert(0, 'DOI', dfs[0]['DOI'])
discrepancies = df_ds[~df_ds['agreement']].drop(columns=['agreement'])

print(f"Records with discrepancies: {len(discrepancies)}")
discrepancies

Records with discrepancies: 2


,run_1,run_2,run_3,run_4,run_5
5,diving48;finegym;mtl-aqa;smart,diving48;finegym;mtl-aqa;smart,diving48;finegym;mtl-aqa;smart,diving48;finegym;mtl-aqa;smart,mtl-aqa;smart
7,gymnastic vaults;mtl-aqa;olympics diving,gymnastic vaults;mtl-aqa;olympics diving,gymnastic vaults;mtl-aqa;olympics diving,mtl-aqa,gymnastic vaults;mtl-aqa;olympics diving
